# Tutorial -  VaR Paramétrico
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

In [ ]:
pip install arch

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import seaborn as sns

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera,shapiro
from scipy.optimize import brentq
from scipy import stats

# ARCH model
from arch import arch_model

# Engel test for ARCH effects
from statsmodels.stats.diagnostic import het_arch


## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción seleccionada:
ticker = "BTC-USD" # cambiar por el ticker de la acción deseada
start_date = "2018-01-01"
df = yfin.download(ticker, start = start_date, multi_level_index=False)
df

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Utiliza la librería mpf para hacer un gráfico de velas japonesas de la acción

In [ ]:
mpf.plot(df,type='candle', volume=True,figratio=(19,8),style='yahoo',title=ticker)

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(15, 12), gridspec_kw={'height_ratios': [3, 1.5, 1.5]})
fig.suptitle('Análisis de Retornos Logarítmicos', fontsize=16, fontweight='bold', y=0.98)

# --- Panel 1: Serie temporal de retornos ---
ax1 = axes[0]
returns = df['Log Returns'].dropna()

# Colorear positivos/negativos
ax1.bar(returns.index, returns.where(returns >= 0), color='#2ecc71', alpha=0.7, width=1, label='Positivo')
ax1.bar(returns.index, returns.where(returns < 0),  color='#e74c3c', alpha=0.7, width=1, label='Negativo')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.axhline(returns.mean(), color='navy', linewidth=1.2, linestyle='--', label=f'Media: {returns.mean():.4f}')
ax1.axhline(returns.mean() + 2*returns.std(), color='orange', linewidth=1, linestyle=':', label=f'±2σ: {2*returns.std():.4f}')
ax1.axhline(returns.mean() - 2*returns.std(), color='orange', linewidth=1, linestyle=':')

ax1.set_title('Retornos Logarítmicos Diarios', fontsize=13)
ax1.set_ylabel('Retorno Log')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_facecolor('#f8f9fa')
ax1.grid(True, alpha=0.3)

# --- Panel 2: Retorno acumulado ---
ax2 = axes[1]
cumulative = returns.cumsum()
ax2.plot(cumulative.index, cumulative, color='#2c3e50', linewidth=1.5)
ax2.fill_between(cumulative.index, cumulative, 0,
                 where=(cumulative >= 0), color='#2ecc71', alpha=0.3)
ax2.fill_between(cumulative.index, cumulative, 0,
                 where=(cumulative < 0),  color='#e74c3c', alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Retorno Logarítmico Acumulado', fontsize=13)
ax2.set_ylabel('Retorno Acumulado')
ax2.set_facecolor('#f8f9fa')
ax2.grid(True, alpha=0.3)

# --- Panel 3: Distribución (histograma + KDE) ---
ax3 = axes[2]
ax3.hist(returns, bins=60, density=True, color='steelblue', alpha=0.6, edgecolor='white')

from scipy.stats import norm
mu, sigma = returns.mean(), returns.std()
x = np.linspace(returns.min(), returns.max(), 300)
ax3.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')
ax3.axvline(mu, color='navy', linestyle='--', linewidth=1.2)
ax3.set_title('Distribución de Retornos', fontsize=13)
ax3.set_xlabel('Retorno Log')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_facecolor('#f8f9fa')
ax3.grid(True, alpha=0.3)

# Estadísticas como texto
stats_text = (f"Obs: {len(returns):,}  |  Media: {mu:.4f}  |  σ: {sigma:.4f}  |  "
              f"Skew: {returns.skew():.3f}  |  Kurt: {returns.kurtosis():.3f}  |  "
              f"Min: {returns.min():.4f}  |  Max: {returns.max():.4f}")
fig.text(0.5, 0.01, stats_text, ha='center', fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', alpha=0.8))

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Normalidad de los retornos

### 6.1. Histogram

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(df['Log Returns'])))

plt.hist(df['Log Returns'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Log-Returns')

# Show the plot
plt.show()

### 6.2. Q-Q Plot

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
sm.qqplot(df['Log Returns'], stats.norm, loc=mean, scale=std_dev, line='45')
# Se agrega un título a la gráfica
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 6.3. P-P Plot

In [ ]:
# Se crea una PP plot calculando las probabilidades empíricas y teóricas
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
fig, ax = plt.subplots()
# Se calculan las probabilidades empíricas
p = np.arange(1, n + 1) / n - 0.5 / n
# Se calculan las probabilidades teóricas
pp = np.sort(stats.norm.cdf(df['Log Returns'],loc=mean, scale=std_dev ))
sns.scatterplot(x=pp, y=p, color='blue', edgecolor='blue', ax=ax)
ax.set_title('P-P plot')
ax.set_xlabel('Theoretical Probabilities')
ax.set_ylabel('Sample Probabilities')
ax.margins(x=0, y=0)
# Se dibuja la línea roja de 45°
plt.plot(np.linspace(0, 1.01), np.linspace(0, 1.01), 'r', lw=2)
# Se muestra la gráfica
plt.show()

### 6.4. Jarque-Bera test

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(df['Log Returns'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
# Crear el autocorrelograma
plot_acf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2

In [ ]:
# Crear el autocorrelograma de retornos al cuadrado
plot_acf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.3. Ljung-Box test

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 5
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## 8. ARCH effects

### 8.2. LJung - Box Test

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 6
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns^2'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## 9. GARCH model

### 9.1. GARCH model

$ \sigma_{t}^{2} = \alpha_{0} + \sum^{q}_{j=1} \alpha_j ϵ_{t-j}^{2} +  \sum^{p}_{i=1} \beta_i \sigma_{t-i}^{2} $

In [ ]:
# Create the ARCH model
model = arch_model(df['Log Returns'], mean='Zero', vol='GARCH', q=1, p=1, rescale=False)

# Fit the model
results = model.fit()

# Print the results summary
print(results.summary())

In [ ]:
params = pd.DataFrame(results.params)
params

In [ ]:
# get residuals
e = pd.DataFrame(results.std_resid)
e['resid^2'] = e['std_resid']**2
e

### 9.2. Histogram of Residuals

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(e['std_resid'])))

plt.hist(e['std_resid'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')

# Show the plot
plt.show()

### 9.3. Q-Q plot - Residuals

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.qqplot(line="45")
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 9.4. P-P plot - Residuals

In [ ]:
# Se crea una P-PQ plot
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.ppplot(line="45")
plt.title("P-P Plot")
# Se muestra la gráfica
plt.show()

### 9.5. Autocorrelation - Residuals

In [ ]:
# Crear el autocorrelograma
plot_acf(e['std_resid'], lags = 21)
plt.show()

In [ ]:
# Crear el autocorrelograma
plot_acf(e['resid^2'], lags = 21)
plt.show()

### 9.6. Jarque Bera Test - Residuals

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(e['std_resid'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

### 9.7. Box-Ljung test - Squared.Residuals

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 5
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(e['resid^2'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

### 9.8. Volatility - Forecast

In [ ]:
fig = results.plot(annualize="D")

In [ ]:
df['Volatility'] = results.conditional_volatility*math.sqrt(252)
# Gráfico de la volatilidad
plt.figure(figsize=(15,8))
plt.plot(df['Volatility'], color = 'orange')
plt.title('Volatility - GARCH(1,1)')
plt.xlabel('Fecha')
plt.ylabel('Volatility')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(15, 13), gridspec_kw={'height_ratios': [3, 1.5, 1.5]})
fig.suptitle('Volatilidad Condicional — GARCH(1,1)', fontsize=16, fontweight='bold', y=0.98)

vol = df['Volatility'].dropna()

# --- Panel 1: Volatilidad condicional ---
ax1 = axes[0]
ax1.plot(vol.index, vol, color='#e67e22', linewidth=1.2, label='Vol. Condicional (anualizada)')

# Bandas de percentiles
p25, p75 = vol.quantile(0.25), vol.quantile(0.75)
p10, p90 = vol.quantile(0.10), vol.quantile(0.90)
mean_vol  = vol.mean()

ax1.axhline(mean_vol, color='navy',  linewidth=1.2, linestyle='--', label=f'Media: {mean_vol:.2%}')
ax1.axhline(p75,      color='gray',  linewidth=0.9, linestyle=':',  label=f'P25-P75: {p25:.2%} – {p75:.2%}')
ax1.axhline(p25,      color='gray',  linewidth=0.9, linestyle=':')
ax1.fill_between(vol.index, p25, p75, alpha=0.12, color='orange', label='Rango intercuartílico')

# Marcar máximos históricos
top5 = vol.nlargest(5)
ax1.scatter(top5.index, top5.values, color='red', zorder=5, s=40, label='Top 5 picos')
for date, value in top5.items():
    ax1.annotate(f'{value:.1%}', xy=(date, value),
                 xytext=(0, 8), textcoords='offset points',
                 fontsize=7.5, color='red', ha='center')

ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax1.set_ylabel('Volatilidad Anualizada')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_facecolor('#f8f9fa')
ax1.grid(True, alpha=0.3)
ax1.set_title('Volatilidad Condicional Anualizada', fontsize=12)

# --- Panel 2: Régimen de volatilidad (heatmap por cuantil) ---
ax2 = axes[1]
# Clasificar régimen: Baja / Media / Alta / Extrema
bins   = [0, p25, mean_vol, p75, np.inf]
labels = ['Baja', 'Media-Baja', 'Media-Alta', 'Alta']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
regimen = pd.cut(vol, bins=bins, labels=labels)

for label, color in zip(labels, colors):
    mask = regimen == label
    ax2.fill_between(vol.index, 0, vol.where(mask), color=color, alpha=0.85, label=label)

ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.set_ylabel('Volatilidad')
ax2.legend(loc='upper left', fontsize=9, ncol=4)
ax2.set_facecolor('#f8f9fa')
ax2.grid(True, alpha=0.3)
ax2.set_title('Régimen de Volatilidad', fontsize=12)

# --- Panel 3: Distribución de la volatilidad ---
ax3 = axes[2]
ax3.hist(vol, bins=60, density=True, color='#e67e22', alpha=0.65, edgecolor='white')

from scipy.stats import lognorm
shape, loc, scale = lognorm.fit(vol, floc=0)
x = np.linspace(vol.min(), vol.max(), 300)
ax3.plot(x, lognorm.pdf(x, shape, loc, scale),
         color='darkred', linewidth=2, label='Log-Normal fit')
ax3.axvline(mean_vol, color='navy', linestyle='--', linewidth=1.2, label=f'Media: {mean_vol:.2%}')
ax3.axvline(p25, color='gray', linestyle=':', linewidth=1)
ax3.axvline(p75, color='gray', linestyle=':', linewidth=1, label=f'IQR: [{p25:.2%}, {p75:.2%}]')
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax3.set_xlabel('Volatilidad Anualizada')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_facecolor('#f8f9fa')
ax3.grid(True, alpha=0.3)
ax3.set_title('Distribución de la Volatilidad', fontsize=12)

# Footer con estadísticas
stats = (f"Obs: {len(vol):,}  |  Media: {mean_vol:.2%}  |  "
         f"Mín: {vol.min():.2%}  |  Máx: {vol.max():.2%}  |  "
         f"σ(vol): {vol.std():.2%}  |  Actual: {vol.iloc[-1]:.2%}")
fig.text(0.5, 0.005, stats, ha='center', fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', alpha=0.8))

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

In [ ]:
print("El último valor de volatilidad es", round(df['Volatility'][-1]*100,4),"%")

## VaR - Paramétrico Normal

### 10.1. Parámetros  

In [ ]:
# cambiar por el nivel de confianza deseado
confidence_level = 0.95

### 10.2. Calculo del VaR

In [ ]:
df['VaR'] = None

In [ ]:
for i in range(0, len(df)):
  # Use iloc for positional slicing to get returns up to day i (exclusive of i for current day's return)
  returns = df['Log Returns'].iloc[0:i]
  confidence = confidence_level

  # Ensure returns is not empty before calculating mean, or handle NaN appropriately
  if not returns.empty:
      mean_val = returns.mean()
  else:
      # If returns is empty (i=0), set mean_val to 0 to avoid NaN, consistent with VaR_2 calculation.
      mean_val = 0

  df.iloc[i, df.columns.get_loc('VaR')] = mean_val + norm.ppf(1 - confidence) * df.iloc[i, df.columns.get_loc('Volatility')] * np.sqrt(1/252)
# VaR - Paramétrico
df['VaR']

In [ ]:
df['VaR_2'] = None
for i in range(0, len(df)):
  returns = df['Log Returns'][0:i]
  #  sorted_returns = np.sort(returns)
  confidence = confidence_level
  df['VaR_2'].iloc[i] = norm.ppf(1 - confidence)*df['Volatility'].iloc[i]*np.sqrt(1/252)
# VaR - Paramétrico
df['VaR_2']

In [ ]:
df['VaR_3'] = None
for i in range(0, len(df)):
  returns = df['Log Returns'][0:i]
  #  sorted_returns = np.sort(returns)
  confidence = confidence_level
  df['VaR_3'][i] = np.sqrt(5)*norm.ppf(1 - confidence)*df['Volatility'][i]*np.sqrt(1/252)
# VaR - Paramétrico
df['VaR_3']

### 10.3. Gráfico del VaR / Paramétrico

In [ ]:
# Gráfico del VaR
plt.figure(figsize=(15,8))
plt.plot(df['VaR'], color = 'red')
plt.title('VaR - Paramétrico')
plt.xlabel('Fecha')
plt.show()

### 10.4 Gráfico Retorno & VaR

In [ ]:
# Gráfico del VaR Simulación Histórica y retornos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.plot(df['VaR'], color = 'red')
plt.plot(df['VaR_2'], color = 'orange')
plt.plot(df['VaR_3'], color = 'c')
plt.title('Retornos Logarítmicos & VaR - Paramétrico Normal')
plt.xlabel('Fecha')
plt.show()

In [ ]:
df['VaR_3'].iloc[-1]

In [ ]:
df['VaR'].iloc[-1]